In [ ]:
# Specify the path to your CSV file containing NIFTI paths
input_csv_path = '/Volumes/OneTouch/01p_Schmahmann_SCA_Atrophy/results/optimzation/optimized_master_list.csv'
sheet = None

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
# Instantiate the PalmPrepararation class
cal_palm = CalvinStatsmodelsPalm(input_csv_path=input_csv_path, output_dir=None, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()
data_df


# 01 - Preprocess Your Data

**Handle NANs**
- Set drop_nans=True is you would like to remove NaNs from data
- Provide a column name or a list of column names to remove NaNs from

In [ ]:
data_df.columns

In [ ]:
drop_list = ["selected_Nifti_File_Path"]

In [ ]:
data_df = cal_palm.drop_nans_from_columns(columns_to_drop_from=drop_list)
data_df

**Drop Row Based on Value of Column**

Define the column, condition, and value for dropping rows
- column = 'your_column_name'
- condition = 'above'  # Options: 'equal', 'above', 'below'

In [ ]:
data_df.columns

Set the parameters for dropping rows

In [ ]:
column = 'Cohort'  # The column you'd like to evaluate
condition = 'not'  # The condition to check ('equal', 'above', 'below', 'not')
value = 'Validation' # The value to drop if found

In [ ]:
# data_df, other_df = cal_palm.drop_rows_based_on_value(column, condition, value)
display(data_df)

This is the Glob-style path to the subfolder containing niftis of interest
- For example, from the base_directory, */tissue_segment_z_scores will look for all subjects, all session folders within subjects, and then check the tissue_segment_z_scores folder. 


In [ ]:
file_column = 'Nifti_File_Path' #'composite_atrophy_path_smoothed' #'roi_path'

In [ ]:
from calvin_utils.file_utils.import_functions import GiiNiiFileImport
dv_df = GiiNiiFileImport(import_path=data_df[file_column], file_pattern=None, file_column=None, process_special_values=True).run()
dv_df

**Extract Subject ID From File Names**
Using the example filenames that have been printed above, please define a general string:
1) Preceding the subject ID.
2) Proceeding the subject ID. 

This Should Often Be Left Default

In [ ]:
preceding_id = '/sub-'
proceeding_id = '_ses'

In [ ]:
from calvin_utils.file_utils.import_functions import GiiNiiFileImport
dv_df = GiiNiiFileImport.splice_colnames(dv_df, preceding_id, proceeding_id)
dv_df

# Load Map to Check Damage Within

Import Region of Interest Masks

In [ ]:
base_directory = '/Volumes/HowExp/resources/atlases/mni_space/aal_atlas/AAL_MNI_V7_fine_rois'
shared_glob_pattern = '*'

In [ ]:
from calvin_utils.file_utils.import_functions import GiiNiiFileImport
iv_df = GiiNiiFileImport(import_path=base_directory, file_pattern=shared_glob_pattern).run()
iv_df

In [ ]:
preceding_id = "AAL_MNI_V7_fine_rois/"
proceeding_id = '.nii'
from calvin_utils.file_utils.import_functions import GiiNiiFileImport
iv_df = GiiNiiFileImport.splice_colnames(iv_df, preceding_id, proceeding_id)
iv_df

Extract Damage Scores Per Region of Interest

In [ ]:
mask_path = '/Users/cu135/Software_Local/calvin_utils_project/circuit_pyper/resources/MNI152_T1_2mm_brain_mask.nii'

In [ ]:
from calvin_utils.neuroimaging_utils.nifti_utils.damage_score_utils import DamageScorer
damage_scorer = DamageScorer(mask_path, dv_df, iv_df)
dmg_df = damage_scorer.calculate_damage_scores('avg_in_target', trace=False)
dmg_df = damage_scorer.sort_dataframes_by_index(dmg_df)
dmg_df

Rename Column in Damage DF

In [ ]:
dmg_df = dmg_df.rename_axis('path').reset_index()
dmg_df = dmg_df.rename(columns={dmg_df.columns[0]: 'Subject'})
dmg_df

Save the Results

In [ ]:
out_path = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/raynor_network_mapping/figures/schmahmann_cerebellar_ataxia/symptom_covariance/atrophy_damage_aal.csv'
dmg_df.to_csv(out_path)

Save Results into Master CSV

In [ ]:
master_path = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/05c_Howard_MetaAlzheimerReview_DBS-TMS_Coordinates/metadata/master_list2.xlsx'
sheet = 'Sheet1'

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
cal_palm = CalvinStatsmodelsPalm(input_csv_path=master_path, output_dir=None, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()
data_df

In [ ]:
# Perform an outer join on dmg_df and data_df using the 'path' column
merged_df = data_df.merge(dmg_df, on='subject', how='outer')
merged_df

In [ ]:
merged_df.to_excel('/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/05c_Howard_MetaAlzheimerReview_DBS-TMS_Coordinates/metadata/master_list2.xlsx', index=False)

In [ ]:
master_path

# Correlation Using Those Damage Scores

In [ ]:
dmg_df.columns

In [ ]:
x_col = 'diagonal'
y_col = 'Percent_Cognitive_Improvement'

In [ ]:
from calvin_utils.statistical_utils.scatterplot import simple_scatter
simple_scatter(merged_df, x_col, y_col, '', 
               x_label="Overlap With Target Network",
               y_label="Cog improve",
               out_dir=None, 
               flip_axes=False)

All done. Enjoy your analyses. 

--Calvin 